# TD 8 : Interface avec Jupyter Notebook
## Moteur de Recherche d'Information Interactive

**Auteurs:** Julien & Martine   
**Date:** 2025-2026

---

## Objectif
Créer une interface interactive complète basée sur un notebook Jupyter pour :
1. Charger un nouveau dataset (discours US)
2. Construire un corpus avec découpage en phrases
3. Intégrer le moteur de recherche TF-IDF
4. Créer une interface graphique avec des widgets ipywidgets

## PARTIE 1 : Chargement et Exploration du Corpus US

### 1.1 : Chargement du dataset

D'abord, chargez le fichier CSV des discours US avec pandas et explorez sa structure.

In [1]:
# Import des librairies nécessaires
import pandas as pd
import sys
import os

# Ajout du chemin du projet pour importer les classes
sys.path.insert(0, os.getcwd())

# 1.1 : Chargement du dataset discours US
try:
    try:
        # test de chargement avec tab comme séparateur (TSV)
        df_discours = pd.read_csv('Data/discours_US.csv', sep='\t', on_bad_lines='skip')
        print("Fichier chargé avec séparateur TAB (TSV) !")
    except:
        # Sinon avec virgule
        df_discours = pd.read_csv('Data/discours_US.csv', on_bad_lines='skip')
        print("Fichier chargé avec séparateur virgule !")
    
    print(f"\nDimensions du dataset : {df_discours.shape}")
    print(f"\nColonnes disponibles : {df_discours.columns.tolist()}")
    print(f"\nAperçu du dataset :\n{df_discours.head(2)}")
    
    # Adapter les noms de colonnes si nécessaire
    if 'speaker' in df_discours.columns and 'text' in df_discours.columns:
        df_discours.rename(columns={'speaker': 'auteur', 'text': 'texte'}, inplace=True)
        print("\nColonnes renommées : 'speaker' → 'auteur', 'text' → 'texte'")
    
except Exception as e:
    print(f"Erreur lors du chargement : {str(e)}")
    

Fichier chargé avec séparateur TAB (TSV) !

Dimensions du dataset : (164, 5)

Colonnes disponibles : ['speaker', 'text', 'date', 'descr', 'link']

Aperçu du dataset :
   speaker                                               text            date  \
0  CLINTON  : I'm getting ready for a lot of things, a lot...  April 12, 2015   
1  CLINTON  [ ] : I'll be graduating in May, and on gradua...  April 14, 2015   

                                               descr  \
0   Video Remarks Announcing Candidacy for President   
1  Remarks in a Question and Answer Session at Ki...   

                                                link  
0  http://www.presidency.ucsb.edu/ws/index.php?pi...  
1  http://www.presidency.ucsb.edu/ws/index.php?pi...  

Colonnes renommées : 'speaker' → 'auteur', 'text' → 'texte'


### 1.2 : Distribution des auteurs

Vérifiez la distribution des auteurs des discours avec la fonction `value_counts()`.

In [2]:
# 1.2 : Distribution des auteurs
print("=" * 70)
print("DISTRIBUTION DES AUTEURS")
print("=" * 70)

# Affichage du nombre de discours par auteur
distribution_auteurs = df_discours['auteur'].value_counts()
print(f"\nNombre de discours par auteur :\n{distribution_auteurs}")

# Affichage en pourcentage
print(f"\n\nPourcentage de discours par auteur :")
pourcentage = (df_discours['auteur'].value_counts(normalize=True) * 100).round(2)
for auteur, pct in pourcentage.items():
    print(f"  {auteur}: {pct}%")

print(f"\nTotal de discours : {len(df_discours)}")
print(f"Nombre d'auteurs uniques : {df_discours['auteur'].nunique()}")


DISTRIBUTION DES AUTEURS

Nombre de discours par auteur :
auteur
CLINTON    93
TRUMP      71
Name: count, dtype: int64


Pourcentage de discours par auteur :
  CLINTON: 56.71%
  TRUMP: 43.29%

Total de discours : 164
Nombre d'auteurs uniques : 2


### 1.3 et 1.4 : Création du Corpus et Découpage en Phrases

Importez la classe Corpus et créez un objet. Les textes étant très longs, découpez les discours en phrases. Chaque phrase constituera un document.

In [3]:
# 1.3 et 1.4 : Création du Corpus avec découpage en phrases

# Import des classes du projet
from Corpus import Corpus
from Document import Document
import re

print("=" * 70)
print("CRÉATION DU CORPUS AVEC DÉCOUPAGE EN PHRASES")
print("=" * 70)

# Creation du corpus
corpus_discours = Corpus("Corpus des Discours US")
print(f"\nCorpus créé : {corpus_discours}")

# Fonction pour découper un texte en phrases
def decouper_en_phrases(texte):
    """
    Découpe un texte en phrases.
    Utilise les points, points d'exclamation et points d'interrogation comme séparateurs.
    """
    # Pattern pour identifier les fins de phrases
    phrases = re.split(r'[.!?]+', texte)
    # Nettoyage des phrases vides et les espaces
    phrases = [p.strip() for p in phrases if p.strip() and len(p.strip()) > 10]
    return phrases

# Ajout des documents au corpus (phrase par phrase)
print("\nAjout des documents au corpus...")
compteur_total = 0

for idx, row in df_discours.iterrows():
    auteur = row['auteur']
    texte = row['texte']
    
    # Découpage en phrases
    phrases = decouper_en_phrases(texte)
    
    for phrase in phrases:
        # Creation  d un document pour chaque phrase
        doc = Document(
            titre=f"Phrase {compteur_total + 1}",
            auteur=auteur,
            date=None,  # Pas de date dans cet exemple
            url="",
            texte=phrase,
            type_doc="discours"
        )
        
        # Ajout au corpus
        corpus_discours.add_document(doc)
        compteur_total += 1

print(f"\n{compteur_total} phrases (documents) ajoutées au corpus")
print(f"\nRésumé du corpus :")
print(f"  - Nombre total de documents : {corpus_discours.ndoc}")
print(f"  - Nombre d'auteurs : {corpus_discours.naut}")
print(f"  - Auteurs : {list(corpus_discours.authors.keys())}")

# Affichage de quelques documents
print(f"\n\nAffichage des 5 premiers documents du corpus :")
print("=" * 70)
corpus_discours.show(n=5)


CRÉATION DU CORPUS AVEC DÉCOUPAGE EN PHRASES
[Singleton] Création de l'instance unique du Corpus
 Corpus 'Corpus des Discours US' initialisé

Corpus créé : Corpus 'Corpus des Discours US': 0 documents, 0 auteurs

Ajout des documents au corpus...

32286 phrases (documents) ajoutées au corpus

Résumé du corpus :
  - Nombre total de documents : 32286
  - Nombre d'auteurs : 2
  - Auteurs : ['CLINTON', 'TRUMP']


Affichage des 5 premiers documents du corpus :
Corpus: Corpus des Discours US
Nombre de documents: 32286
Nombre d'auteurs: 2

 Affichage des 5 premiers documents:

1. Phrase 1 (discours)
2. Phrase 2 (discours)
3. Phrase 3 (discours)
4. Phrase 4 (discours)
5. Phrase 5 (discours)


In [4]:
# Test des fonctions search et concorde

print("\n" + "=" * 70)
print("TEST DES FONCTIONS search ET concorde")
print("=" * 70)

# Test 1 : Recherche simple
print("\n\nTest 1 : Fonction search")
print("-" * 70)
mot_recherche = "america"
passages = corpus_discours.search(mot_recherche)
print(f"\nAffichage des 3 premiers passages contenant '{mot_recherche}' :")
for i, passage in enumerate(passages[:3], 1):
    print(f"\n{i}. ...{passage}...")

# Test 2 : Concordancier
print("\n\nTest 2 : Concordancier")
print("-" * 70)
mot_concorde = "great"
concordancier = corpus_discours.concorde(mot_concorde, taille_contexte=40)
print(f"\nConcordancier pour '{mot_concorde}' (affichage des 5 premières lignes) :")
if len(concordancier) > 0:
    print(concordancier.head(5).to_string())
else:
    print(f"Aucune occurrence de '{mot_concorde}' trouvée.")
    


TEST DES FONCTIONS search ET concorde


Test 1 : Fonction search
----------------------------------------------------------------------
Construction de la chaîne complète (première fois)...
 Chaîne construite : 2745671 caractères
'america' trouvé 1244 fois

Affichage des 3 premiers passages contenant 'america' :

1. ...and stay ahead, because when families are strong, America is strong So I'm hitting the road to earn your vo...

2. ...t we get small businesses starting and growing in America again We have stalled out I was very surprised to...

3. ...s with some hard truths about race and justice in America There is something profoundly wrong when African ...


Test 2 : Concordancier
----------------------------------------------------------------------
Concorde 'great' trouvé 929 fois

Concordancier pour 'great' (affichage des 5 premières lignes) :
                           contexte_gauche motif_trouve                           contexte_droit
0  right, good luck Year and a half That

---

## PARTIE 2 : Utilisation du Moteur de Recherche

### 2.1 et 2.2 : Initialisation et Test du SearchEngine

Importez la classe SearchEngine et testez-la avec plusieurs requêtes.

In [5]:
# 2.1 et 2.2 : Initialisation et test du SearchEngine

from SearchEngine import SearchEngine

print("=" * 70)
print("PARTIE 2 : MOTEUR DE RECHERCHE TFIDF")
print("=" * 70)

print("\n2.1 : Initialisation du SearchEngine...")
# Creation du moteur de recherche
engine = SearchEngine(corpus_discours)

print("\nSearchEngine initialisé !")
print(f"  - Vocabulaire : {len(engine.vocab)} mots uniques")

# Affichage d un aperçu du vocabulaire
print(f"\nAperçu du vocabulaire enrichi (10 mots) :")
for i, (mot, infos) in enumerate(engine.vocab.items()):
    if i >= 10:
        break
    print(f"  {mot}: {infos}")

print("\n\n2.2 : Test du moteur avec plusieurs requêtes")
print("=" * 70)

# Différentes requêtes à tester
requetes_test = [
    "america great",
    "people change",
    "government policy"
]

for requete in requetes_test:
    print(f"\n\nRequête : '{requete}'")
    print("-" * 70)
    
    # Lancement de la recherche
    resultats = engine.search(requete, n_results=3)
    
    if not resultats.empty:
        print(f"Top 3 résultats :")
        print(resultats[['Titre', 'Auteur', 'Score']].to_string(index=False))
    else:
        print("Aucun résultat trouvé.")

PARTIE 2 : MOTEUR DE RECHERCHE TFIDF

2.1 : Initialisation du SearchEngine...
[SearchEngine] Construction du vocabulaire et des matrices...
[SearchEngine] Matrices TF et TF-IDF prêtes !

SearchEngine initialisé !
  - Vocabulaire : 12189 mots uniques

Aperçu du vocabulaire enrichi (10 mots) :
  a: {'id': 0, 'nb_occ': 9122, 'nb_docs': 7436}
  abandon: {'id': 1, 'nb_occ': 9, 'nb_docs': 9}
  abandoned: {'id': 2, 'nb_occ': 22, 'nb_docs': 22}
  abandoning: {'id': 3, 'nb_occ': 3, 'nb_docs': 3}
  abandonment: {'id': 4, 'nb_occ': 2, 'nb_docs': 2}
  abbas: {'id': 5, 'nb_occ': 2, 'nb_docs': 2}
  abc: {'id': 6, 'nb_occ': 1, 'nb_docs': 1}
  abdullah: {'id': 7, 'nb_occ': 1, 'nb_docs': 1}
  abe: {'id': 8, 'nb_occ': 6, 'nb_docs': 5}
  abedin: {'id': 9, 'nb_occ': 1, 'nb_docs': 1}


2.2 : Test du moteur avec plusieurs requêtes


Requête : 'america great'
----------------------------------------------------------------------
Top 3 résultats :
       Titre  Auteur    Score
Phrase 26083 CLINTON 0.736908
Ph

### 2.3 : Intégration de la Barre de Progression (tqdm)

Ajoutez la bibliothèque `tqdm` pour afficher un compteur de progression lors de la recherche.

In [6]:
# 2.3 : Version du SearchEngine avec barre de progression (tqdm)

from tqdm import tqdm
import numpy as np
import math
from scipy.sparse import csr_matrix

print("\n" + "=" * 70)
print("VERSION AMÉLIORÉE : SearchEngine avec TQDM")
print("=" * 70)

class SearchEngineWithProgress(SearchEngine):
    """
    Extension du SearchEngine avec barre de progression tqdm
    """
    
    def search_with_progress(self, query, n_results=5):
        """
        Recherche avec affichage de progression
        """
        # Nettoyer et vectoriser la requête
        mots_requete = self.corpus.nettoyer_texte(query).split()
        vec_requete = np.zeros(len(self.vocab))

        for mot in mots_requete:
            if mot in self.vocab:
                vec_requete[self.vocab[mot]["id"]] += 1

        # Normalisation de la requête
        norme_requete = np.linalg.norm(vec_requete)
        if norme_requete > 0:
            vec_requete = vec_requete / norme_requete

        # Normalisation des vecteurs documents TF-IDF
        mat_norm = self.mat_TFxIDF.copy()

        # Calcul norme document par ligne
        norms = np.sqrt(mat_norm.multiply(mat_norm).sum(axis=1))
        norms = np.array(norms).flatten()

        # Division ligne par ligne avec barre de progression
        print(f"\nNormalisation des vecteurs documents...")
        for i in tqdm(range(mat_norm.shape[0]), desc="Normalisation"):
            if norms[i] > 0:
                mat_norm.data[mat_norm.indptr[i]:mat_norm.indptr[i+1]] /= norms[i]

        # Produit scalaire - similarité cosinus
        scores = mat_norm.dot(vec_requete)

        # Tri décroissant
        indices = np.argsort(scores)[::-1]

        # Préparation DataFrame avec barre de progression
        results = []
        docs_list = list(self.corpus.id2doc.values())

        print(f"\nConstruction des résultats...")
        for i in tqdm(range(min(n_results, len(docs_list))), desc="Résultats"):
            idx = indices[i]
            score = scores[idx]

            if score > 0:
                doc = docs_list[idx]
                results.append({
                    "Titre": doc.titre,
                    "Auteur": doc.auteur,
                    "Date": doc.date,
                    "Source": doc.get_type(),
                    "Score": float(score)
                })

        return pd.DataFrame(results)

# Créer le moteur avec progesssion
engine_progress = SearchEngineWithProgress(corpus_discours)

print("\n\nTest du moteur avec barre de progression...")
print("-" * 70)

requete_demo = "freedom democracy"
print(f"\nRecherche : '{requete_demo}'")

resultats_progress = engine_progress.search_with_progress(requete_demo, n_results=3)

print(f"\n\nRésultats :")
print(resultats_progress.to_string())


VERSION AMÉLIORÉE : SearchEngine avec TQDM
[SearchEngine] Construction du vocabulaire et des matrices...
[SearchEngine] Matrices TF et TF-IDF prêtes !


Test du moteur avec barre de progression...
----------------------------------------------------------------------

Recherche : 'freedom democracy'

Normalisation des vecteurs documents...


Normalisation: 100%|██████████| 32286/32286 [00:00<00:00, 219401.60it/s]



Construction des résultats...


Résultats: 100%|██████████| 3/3 [00:00<00:00, 22753.91it/s]



Résultats :
          Titre   Auteur  Date    Source     Score
0  Phrase 20185  CLINTON  None  discours  0.490687
1   Phrase 1472  CLINTON  None  discours  0.473001
2  Phrase 29148    TRUMP  None  discours  0.413598


---

## PARTIE 3 : Interface Interactive avec Widgets

Créez une interface visuelle basée sur des widgets ipywidgets pour interroger le corpus de manière interactive.

In [7]:
# PARTIE 3 : Interface Interactive avec Widgets

import ipywidgets as widgets
from IPython.display import display, clear_output

print("=" * 70)
print("PARTIE 3 : INTERFACE INTERACTIVE AVEC WIDGETS")
print("=" * 70)

# 3.1 et 3.2 : Création des widgets de base

# Titre (widgets.HTML en gras et centré)
label_titre = widgets.HTML("<div style='text-align:center;'><b style='font-size:20px;'>Moteur de Recherche</b></div>")

# Champ de saisie pour les mots-clés
text_requete = widgets.Text(
    value='mot1 mot2 mot3',
    placeholder='Entrez vos mots-clés...',
    description='Mots clés :',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='80%')
)

# Slider pour le nombre d'articles à extraire
slider_resultats = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description='Nombre :',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='50%')
)

# Affichage initial
print("\n3.1 et 3.2 : Création des widgets de base")
print("=" * 70)
print("\nWidgets créés :")
print("  - HTML : Titre (gras, centré)")
print("  - Text : Champ de requête")
print("  - IntSlider : nombre d'articles à extraire")

# Organisation verticale et centrée : chaque élément sur sa propre ligne
# On expose aussi les widgets individuellement pour que l'interface soit claire

# 3.3 : Création de l'objet Output
output_resultats = widgets.Output(
    layout=widgets.Layout(
        border='1px solid black',
        padding='10px',
        width='80%',
        height='300px',
        overflow_y='auto'
    )
)

print("\n3.3 : Création de l'objet Output")
print("=" * 70)
print("Output widget créé pour afficher les résultats")

# 3.4 et 3.5 : Bouton et fonction de clique

def clique_bouton(b):
    """
    Fonction appelée lors du clique sur le bouton.
    Lance la recherche et affiche les résultats.
    """
    # Vider la sortie précédente
    output_resultats.clear_output()
    
    with output_resultats:
        # Récupération des valeurs des widgets
        requete = text_requete.value
        n_resultats = slider_resultats.value
        
        if not requete or requete.strip() == "":
            print("Veuillez entrer une requête !")
            return
        
        print(f"\nRecherche en cours pour : '{requete}'")
        print(f"Nombre d'articles à extraire : {n_resultats}")
        print("-" * 70)
        
        try:
            # Lancement de la recherche avec l'engine
            resultats = engine.search(requete, n_results=n_resultats)
            
            if resultats.empty:
                print("\nAucun résultat trouvé pour cette requête.")
            else:
                print(f"\n{len(resultats)} résultat(s) trouvé(s) !\n")
                
                # Affichage détaillé des résultats
                for idx, row in resultats.iterrows():
                    print(f"Résultat {idx + 1}")
                    print(f"  Titre : {row['Titre']}")
                    print(f"  Auteur : {row['Auteur']}")
                    print(f"  Source : {row['Source']}")
                    print(f"  Score de similarité : {row['Score']:.4f}")
                    print()
                    
        except Exception as e:
            print(f"\nErreur lors de la recherche : {str(e)}")

# Création du bouton
button_recherche = widgets.Button(
    description='Rechercher',
    button_style='info',
    tooltip='Cliquez pour lancer la recherche',
    icon='search',
    layout=widgets.Layout(width='30%')
)

# Attacher la fonction au clique
button_recherche.on_click(clique_bouton)

print("\n3.4 et 3.5 : Création du bouton et fonction de clique")
print("=" * 70)
print("Bouton 'Rechercher' créé")
print("Fonction clique_bouton définie et attachée au bouton")

# Organisation finale de l'interface : chaque élément sur une ligne et centré
interface_complete = widgets.VBox([
    label_titre,
    text_requete,
    slider_resultats,
    button_recherche,
    output_resultats
], layout=widgets.Layout(align_items='center', width='100%', gap='10px'))

print("\n\n3.6 : Affichage de l'interface")
print("=" * 70)
print("\nInterface interactive créée et prête à l'emploi !")
print("\nUtilisez l'interface ci-dessous pour effectuer vos recherches.")

# Affichage de l'interface
display(interface_complete)

PARTIE 3 : INTERFACE INTERACTIVE AVEC WIDGETS

3.1 et 3.2 : Création des widgets de base

Widgets créés :
  - HTML : Titre (gras, centré)
  - Text : Champ de requête
  - IntSlider : nombre d'articles à extraire

3.3 : Création de l'objet Output
Output widget créé pour afficher les résultats

3.4 et 3.5 : Création du bouton et fonction de clique
Bouton 'Rechercher' créé
Fonction clique_bouton définie et attachée au bouton


3.6 : Affichage de l'interface

Interface interactive créée et prête à l'emploi !

Utilisez l'interface ci-dessous pour effectuer vos recherches.


---

## 3.7 : Extension Optionnelle - Filtres Avancés

Ajoutez des filtres pour affiner la recherche par auteur et/ou date.

In [ ]:
# 3.7 : Interface avec Filtres Avancés

print("\n" + "=" * 70)
print("3.7 : EXTENSION - INTERFACE AVEC FILTRES AVANCÉS")
print("=" * 70)

# Récupération de la liste des auteurs
# (corpus_discours.authors est un dictionnaire auteur -> liste docs)
auteurs_list = list(corpus_discours.authors.keys())

# Création des widgets supplémentaires pour les filtres
label_filtres = widgets.HTML("<div style='text-align:center; font-weight:600;'>Filtres Avancés</div>")

# Dropdown pour sélectionner un auteur
dropdown_auteur = widgets.Dropdown(
    options=['Tous'] + auteurs_list,
    value='Tous',
    description='Auteur :',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='50%')
)

# Checkbox pour activer/désactiver le filtre
checkbox_activer_filtres = widgets.Checkbox(
    value=False,
    description='Activer les filtres',
    indent=False,
    layout=widgets.Layout(width='50%')
)

# Organisation des filtres : chaque élément sur sa propre ligne et centré
vbox_filtres_inner = widgets.VBox([checkbox_activer_filtres, dropdown_auteur], layout=widgets.Layout(align_items='center', width='100%'))
vbox_filtres = widgets.VBox([label_filtres, vbox_filtres_inner], layout=widgets.Layout(align_items='center', width='100%'))

# Fonction de recherche améliorée avec filtres
def clique_bouton_filtre(b):
    """
    Fonction améliorée avec support des filtres.
    """
    # Vider la sortie précédente
    output_resultats.clear_output()
    
    with output_resultats:
        # Récupération des valeurs
        requete = text_requete.value
        n_resultats = slider_resultats.value
        activer_filtres = checkbox_activer_filtres.value
        auteur_selectionne = dropdown_auteur.value if activer_filtres else 'Tous'
        
        if not requete or requete.strip() == "":
            print("Veuillez entrer une requête !")
            return
        
        print(f"\nRecherche en cours pour : '{requete}'")
        print(f"Nombre d'articles à extraire : {n_resultats}")
        
        if activer_filtres and auteur_selectionne != 'Tous':
            print(f"Filtre auteur : {auteur_selectionne}")
        
        print("-" * 70)
        
        try:
            # Lancement de la recherche
            resultats = engine.search(requete, n_results=n_resultats * 2)
            
            # Appliquer le filtre auteur si nécessaire
            if activer_filtres and auteur_selectionne != 'Tous':
                resultats = resultats[resultats['Auteur'] == auteur_selectionne]
                # Garder seulement n_resultats
                resultats = resultats.head(n_resultats)
            else:
                resultats = resultats.head(n_resultats)
            
            if resultats.empty:
                print("\nAucun résultat trouvé avec les critères spécifiés.")
            else:
                print(f"\n{len(resultats)} résultat(s) trouvé(s) !\n")
                
                # Affichage détaillé
                for idx, row in resultats.iterrows():
                    print(f"Résultat {idx + 1}")
                    print(f"  Titre : {row['Titre']}")
                    print(f"  Auteur : {row['Auteur']}")
                    print(f"  Source : {row['Source']}")
                    print(f"  Score : {row['Score']:.4f}")
                    print()
                    
        except Exception as e:
            print(f"\nErreur : {str(e)}")

# Création du bouton pour la recherche avec filtres
button_recherche_filtre = widgets.Button(
    description='Rechercher',
    button_style='success',
    tooltip='Recherche avec filtres optionnels',
    icon='search',
    layout=widgets.Layout(width='30%')
)

button_recherche_filtre.on_click(clique_bouton_filtre)

# Interface complète avec filtres : éléments empilés et centrés
interface_complete_filtre = widgets.VBox([
    label_titre,
    text_requete,
    slider_resultats,
    vbox_filtres,
    button_recherche_filtre,
    output_resultats
], layout=widgets.Layout(align_items='center', width='100%', gap='10px'))

print("\nInterface avec filtres avancés créée !")
print("\nWidgets ajoutés :")
print("  Checkbox : Activer les filtres")
print("  Dropdown : Sélectionner un auteur")
print("\nAffichage de l'interface complète :")
print("=" * 70)

display(interface_complete_filtre)


3.7 : EXTENSION - INTERFACE AVEC FILTRES AVANCÉS

Interface avec filtres avancés créée !

Widgets ajoutés :
  Checkbox : Activer les filtres
  Dropdown : Sélectionner un auteur

Affichage de l'interface complète :




## Conclusion et Résumé

### Ce que nous avons réalisé

**Partie 1** : Chargement du dataset, exploration et structuration du corpus  
**Partie 2** : Intégration du moteur de recherche TF-IDF avec tqdm  
**Partie 3** : Interface interactive complète avec widgets ipywidgets  
**Bonus** : Filtres avancés par auteur

### Points clés

- **Réutilisation du code** : Classes Corpus, Document et SearchEngine importées des TD précédents
- **Découpage en phrases** : Chaque phrase devient un document indépendant
- **Barre de progression** : tqdm affiche l'avancement du traitement
- **Widgets ipywidgets** : VBox, HBox, Label, Text, IntSlider, Button, Output, Dropdown, Checkbox
- **Filtres avancés** : Possibilité de filtrer par auteur